In [2]:
!pip install streamlit
!pip install pyngrok
!pip install python-docx
!pip install -U transformers
!pip install -q trl
!pip install trl==0.11.3
!pip install -q accelerate>=1.8.0
!pip install -q bitsandbytes>=0.46.1
!pip uninstall pyarrow datasets -y
!pip install -q datasets
!pip install pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 52.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 56.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.1/11.1 MB 112.8 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.10.1
    Uninstalling transformers-5.10.1:
      Successfully uninstalled transformers-5.10.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 18.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.6/316.6 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 185.2/185.2 kB 22.6 MB/s eta 0:00:00
  Attempting uninstall: trl
    Found existing installation: trl 1.5.1
    Uninstalling trl-1.5.1:
  

In [4]:
import streamlit as st
import time
import random
import re
import pandas as pd
import numpy as np
import torch
import gc
import os
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import openai
from datasets import Dataset
from pyngrok import ngrok
import threading
import subprocess

torch.cuda.empty_cache()
gc.collect()

60

In [6]:
%%writefile app.py
import streamlit as st
import time
import random
import re
import pandas as pd
import numpy as np
import torch
import gc
import os
from transformers import AutoTokenizer, BitsAndBytesConfig, AutoModelForCausalLM
import openai

st.set_page_config(
    page_title="Автоматическре кодирование интервью",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.title("Анализ интервью с помощью AI")
st.markdown("### Генерация тематических кодов и цитат из текстов интервью")
st.markdown("---")

if 'model_loaded' not in st.session_state:
    st.session_state.model_loaded = False
if 'results' not in st.session_state:
    st.session_state.results = []

Overwriting app.py


In [7]:
%%writefile -a app.py

with st.sidebar:
    st.header("Настройки")

    st.subheader("Параметры генерации")

    max_new_tokens = st.slider(
        "Макс. токенов",
        min_value=512,
        max_value=2048,
        value=1024,
        step=128,
        help="Максимальное количество генерируемых токенов"
    )

    num_beams = st.slider(
        "Beam search",
        min_value=1,
        max_value=5,
        value=2,
        step=1,
        help="Ширина beam search (1 = жадный поиск)"
    )

    temperature = st.slider(
        "Температура",
        min_value=0.0,
        max_value=1.0,
        value=0.0,
        step=0.1,
        help="0.0 = детерминированная генерация, выше = больше креативности"
    )

    st.markdown("---")
    st.markdown("### О приложении")
    st.info(
        "Это приложение выполняет тематическое кодирование интервью "
        "с использованием модели Qwen2.5-7B-Instruct.\n\n"
        "**Формат вывода:**\n"
        "- Общие коды (темы)\n"
        "- Конкретные коды (подтемы)\n"
        "- Соответствующие цитаты"
    )

    st.markdown("---")
    st.caption("© 2024 | Кодирование интервью с AI")

Appending to app.py


In [8]:
%%writefile -a app.py

@st.cache_resource
def load_model_and_tokenizer():
    """Загрузка модели и токенизатора"""
    with st.spinner("Загрузка модели Qwen2.5-7B-Instruct..."):
        base_model = "Qwen/Qwen2.5-7B-Instruct"

        tokenizer = AutoTokenizer.from_pretrained(
            base_model,
            trust_remote_code=True
        )
        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=torch.bfloat16
        )

        model = AutoModelForCausalLM.from_pretrained(
            base_model,
            quantization_config=bnb_config,
            device_map="auto",
            torch_dtype=torch.bfloat16
        )

        model.eval()

        st.success("Модель успешно загружена!")
        return model, tokenizer

def clear_cuda_cache():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()
        gc.collect()

Appending to app.py


In [9]:
%%writefile -a app.py

def build_prompt_h4(example, tokenizer):
    instruction = f"""You are an expert in interview analysis. Your task is to highlight the thematic codes in the interview text and corresponding quotes.

Act step by step:

1. Carefully review the output format shown in the example below. Remember that each code block must contain "Общий код" (General code), then Quote, then "Конкретный код" (Specific code). The quote must be verbatim and enclosed in quotation marks.
2. Read the interview transcript and the topic. Identify all fragments (quotes) that relate to the interview topic.
3. Group the quotes by general themes — these will be the "Общий код" (General codes). For each general theme, come up with a short name.
4. Within each general code, identify specific meaning aspects — these will be the "конкретный код" (Specific codes). The names of specific codes should reflect the essence of the quote.
5. Generate the answer strictly following the format from the example. Do not add any explanations, do not write words like 'Step 1', 'Step 2' — only the final blocks of codes and quotes.

Format of an output (consists of several general codes, each followed by quotes and specific codes):
**Общий код 1: <generate general code 1>**
"<quote text>" - **<generate specific code 1> (Конкретный код)**
"<quote text>" - **<generate specific code 2> (Конкретный код)**
**Общий код 2: <generate general code 2>**
"<quote text>" - **<generate specific code> (Конкретный код)**
**Общий код <general code number>: <generate general code>**
"<quote text>" - **<generate specific code> (Конкретный код)**
and so on. You choose the number of general and specific codes.

Example of a topic:
ПОКОЛЕНЧЕСКАЯ ИДЕНТИЧНОСТЬ, ЖИЗНЕННЫЕ ВЫБОРЫ И УСТОЙЧИВОСТЬ РОССИЙСКОЙ МОЛОДЕЖИ

Example of a general code related to the topic (pay attention to the structure):
**Общий код 1: Поколенческие характеристики, ценности и жизненные ориентиры**
"Мне кажется, большинство особо не стремятся там вот срочно, прямо сейчас там жениться, замуж, там детей и так далее. Вот. То есть как-то больше сосредоточены даже не на карьере, а на себе, на том, чтобы сложить все для себя вот так, как хочется, да. То есть не просто чтобы там выйти замуж, а чтобы выйти замуж вот по любви, чтобы все было идеально. Вот на какой-то такой идеальности что ли." - **Фокус на самореализации и качественных отношениях (конкретный код)**
"У нашего поколения все-таки все по-другому. Нам не дадут квартиру просто так. Нам не обязательно так просто получить место там где-то на работе и так далее, да. Но при этом у нас гораздо больше возможностей в плане, как сказать, чему-то научиться новому, куда-то поехать, что-то посмотреть, составить свое мнение, там высказать свое мнение даже." - **Осознание свободы выбора и новых возможностей (конкретный код)**
"У детей нынешних у них как будто меньше табу что ли. [...] они спокойно со мной могла поговорить на какие-то откровенные темы, которые мне в ее возрасте, я тоже задумывалась об этом, у меня тоже было какое-то мнение, но я боялась об этом говорить со взрослыми, потому что это было табуировано. [...] Поэтому у них мне кажется растет какое-то более свободное поколение что ли." - **Сравнение с младшим поколением: свобода от табу (конкретный код)**
"Мне кажется, что поколение у нас достаточно трудолюбивое при этом как бы. То есть если человек чего-то хочет добиться в карьерной сфере, ну, человек действительно может приложить там все усилия и добиться этого. Вот. Потому что возможностей супер много сейчас." - **Трудолюбие и вера в возможности (конкретный код)**

Now you should do the markup for the interview according to the plan. Important: The answer should contain only codes and quotes, without unnecessary words and repetitions.
The quotes must be strictly from the text. Give the answer in Russian.
"""

    user_content = f"""Тема интервью:
{example['topic']}

Текст интервью:
{example['transcript']}

Пожалуйста, выполни разметку интервью, каждый общий код в указанном формате:
**Общий код <generate general code>: <general code name>**
"<quote text>" - **<generate specific code> (Конкретный код)**"""

    messages = [
        {"role": "system", "content": instruction},
        {"role": "user", "content": user_content},
    ]
    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    return prompt

Appending to app.py


In [10]:
%%writefile -a app.py

def generate_code(model, tokenizer, prompt, max_new_tokens=1024, num_beams=2, temperature=0.0):
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=3600)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        if temperature > 0:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_p=0.95,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )
        else:
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=num_beams,
                early_stopping=True,
                pad_token_id=tokenizer.pad_token_id,
                eos_token_id=tokenizer.eos_token_id
            )

    generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
    return generated

Appending to app.py


In [11]:
%%writefile -a app.py

def main():
    if not st.session_state.model_loaded:
        try:
            model, tokenizer = load_model_and_tokenizer()
            st.session_state.model = model
            st.session_state.tokenizer = tokenizer
            st.session_state.model_loaded = True
        except Exception as e:
            st.error(f"Ошибка загрузки модели: {e}")
            st.stop()
    else:
        model = st.session_state.model
        tokenizer = st.session_state.tokenizer

    tab1, tab2 = st.tabs(["Кодирование интервью", "Пакетное кодирование"])

    with tab1:
        st.header("Кодирование одного интервью")
        st.markdown("---")

        with st.expander("Пример формата вывода", expanded=False):
            st.markdown("""
            **Общий код 1: Поколенческие характеристики**
            "Мне кажется, большинство не стремятся срочно жениться..." - **Фокус на самореализации (Конкретный код)**
            "У нашего поколения все по-другому..." - **Осознание свободы выбора (Конкретный код)**

            **Общий код 2: Ценности и ориентиры**
            "Человек может приложить усилия и добиться..." - **Трудолюбие и вера в возможности (Конкретный код)**
            """)

        with st.form("analysis_form"):
            topic = st.text_area(
                "**Тема интервью**",
                height=100,
                placeholder="Введите тему интервью здесь...",
                help="Основная тема, которую нужно проанализировать"
            )

            transcript = st.text_area(
                "**Текст интервью**",
                height=300,
                placeholder="Введите текст интервью для анализа...",
                help="Полный текст интервью, который нужно закодировать"
            )

            col1, col2, col3 = st.columns([1, 2, 1])
            with col2:
                submitted = st.form_submit_button("Выполнить кодирование", type="primary", use_container_width=True)

        if submitted:
            if not topic or not transcript:
                st.error("Пожалуйста, заполните тему и текст интервью!")
            else:
                with st.spinner("Генерация кодов и цитат..."):
                    example = {
                        'topic': topic,
                        'transcript': transcript
                    }

                    prompt = build_prompt_h4(example, tokenizer)

                    generated_codes = generate_code(
                        model, tokenizer, prompt,
                        max_new_tokens=max_new_tokens,
                        num_beams=num_beams,
                        temperature=temperature
                    )

                    st.success("Кодирование завершено!")

                    st.markdown("### Результаты кодирования")
                    st.markdown(generated_codes)

                    col1, col2 = st.columns(2)
                    with col1:
                        st.download_button(
                            label="Скачать результат (TXT)",
                            data=generated_codes,
                            file_name=f"coding_{int(time.time())}.txt",
                            mime="text/plain",
                            use_container_width=True
                        )

                    with col2:
                        import json
                        json_result = json.dumps({
                            "topic": topic,
                            "transcript": transcript[:500] + "...",
                            "generated_codes": generated_codes,
                            "timestamp": time.time()
                        }, ensure_ascii=False, indent=2)

                        st.download_button(
                            label="Скачать результат (JSON)",
                            data=json_result,
                            file_name=f"coding_{int(time.time())}.json",
                            mime="application/json",
                            use_container_width=True
                        )

                    with st.expander("Показать промпт (для отладки)"):
                        st.code(prompt, language="markdown")

    with tab2:
        st.header("Пакетное кодирование интервью")
        st.markdown("---")

        st.info(
            "Загрузите CSV файл с колонками: **topic**, **transcript**\n\n"
            "Результат будет сохранен в CSV файл с добавленной колонкой **generated_codes**"
        )

        uploaded_file = st.file_uploader(
            "Выберите CSV файл",
            type="csv",
            help="Файл должен содержать колонки: topic (тема), transcript (текст интервью)"
        )

        if uploaded_file is not None:
            df = pd.read_csv(uploaded_file)
            st.write("### Предпросмотр данных")
            st.dataframe(df.head(), use_container_width=True)

            required_cols = ['topic', 'transcript']
            missing_cols = [col for col in required_cols if col not in df.columns]

            if missing_cols:
                st.error(f"Отсутствуют необходимые колонки: {missing_cols}")
            else:
                st.write(f"**Всего записей:** {len(df)}")

                col1, col2, col3 = st.columns([1, 2, 1])
                with col2:
                    batch_button = st.button(
                        "Запустить пакетное кодирование",
                        type="primary",
                        use_container_width=True
                    )

                if batch_button:
                    results = []
                    progress_bar = st.progress(0)
                    status_text = st.empty()

                    for idx, row in df.iterrows():
                        status_text.text(f"Обработка {idx+1}/{len(df)}...")

                        example = {
                            'topic': row['topic'],
                            'transcript': row['transcript']
                        }

                        prompt = build_prompt_h4(example, tokenizer)
                        generated = generate_code(
                            model, tokenizer, prompt,
                            max_new_tokens=max_new_tokens,
                            num_beams=num_beams,
                            temperature=temperature
                        )

                        result = {
                            'index': idx,
                            'topic': row['topic'],
                            'transcript_preview': row['transcript'][:200] + "...",
                            'generated_codes': generated
                        }

                        results.append(result)
                        progress_bar.progress((idx + 1) / len(df))

                        if (idx + 1) % 5 == 0:
                            clear_cuda_cache()

                    status_text.text("Кодирование завершено!")
                    progress_bar.empty()

                    results_df = pd.DataFrame(results)

                    st.markdown("---")
                    st.write("### Результаты кодирования")
                    st.dataframe(results_df, use_container_width=True)

                    col1, col2 = st.columns(2)
                    with col1:
                        csv = results_df.to_csv(index=False)
                        st.download_button(
                            label="Скачать результаты (CSV)",
                            data=csv,
                            file_name=f"batch_coding_{int(time.time())}.csv",
                            mime="text/csv",
                            use_container_width=True
                        )

                    with col2:
                        import json
                        json_results = results_df.to_json(orient='records', force_ascii=False)
                        st.download_button(
                            label="Скачать результаты (JSON)",
                            data=json_results,
                            file_name=f"batch_coding_{int(time.time())}.json",
                            mime="application/json",
                            use_container_width=True
                        )

                    with st.expander("Пример результатов", expanded=False):
                        if len(results_df) > 0:
                            st.markdown(f"**Пример {results_df.iloc[0]['topic']}:**")
                            st.code(results_df.iloc[0]['generated_codes'][:500] + "...")

if __name__ == "__main__":
    main()

Appending to app.py


In [12]:
def run_streamlit():
    os.system('pkill -f ngrok')
    os.system('streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &')
    time.sleep(5)
    print("Streamlit сервер запущен на порту 8501")

run_streamlit()

NGROK_TOKEN = "3DwzqhWEX9aUEF5spTUYpBMceBa_7b6JgtJfqNc9JCqZeX3Vd"

if NGROK_TOKEN:
    from pyngrok import ngrok
    ngrok.kill()
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(8501)
    print("\n" + "="*60)
    print("Streamlit приложение запущено!")
    print("="*60)
    print(f"\nПубличный URL: {public_url}")
    print("\nОткройте эту ссылку в браузере для доступа к приложению")
    print("Ссылка будет активна пока работает Colab сессия")
else:
    print("\n" + "="*60)
    print("Streamlit приложение запущено локально!")
    print("="*60)
    print("\nЛокальный доступ: http://localhost:8501")
    print("Для публичного доступа:")
    os.system('pkill -f streamlit')
    print("   1. Зарегистрируйтесь на https://ngrok.com")
    print("   2. Получите токен авторизации")
    print("   3. Вставьте токен в переменную NGROK_TOKEN")
    print("   4. Запустите ячейку заново")

print("\n" + "="*60)
print("ИНСТРУКЦИЯ ПО ИСПОЛЬЗОВАНИЮ")
print("="*60)
print("\n1) Дождитесь загрузки модели (2-3 минуты)")
print("2) Для анализа одного интервью:")
print("   - Заполните тему и текст интервью")
print("   - Нажмите 'Выполнить кодирование'")
print("3) Для пакетного анализа:")
print("   - Загрузите CSV файл с колонками: topic, transcript")
print("   - Нажмите 'Запустить пакетное кодирование'")
print("\nВнимание: Модель весит ~4GB, загрузка может занять время")
print("="*60)

if torch.cuda.is_available():
    print(f"\nGPU Memory allocated: {torch.cuda.memory_allocated()/1024**3:.2f} GB")
    print(f"GPU Memory reserved: {torch.cuda.memory_reserved()/1024**3:.2f} GB")

print("\nСессия активна. Нажмите STOP в Colab для остановки.")

Streamlit сервер запущен на порту 8501

Streamlit приложение запущено!

Публичный URL: NgrokTunnel: "https://cassette-sequester-video.ngrok-free.dev" -> "http://localhost:8501"

Откройте эту ссылку в браузере для доступа к приложению
Ссылка будет активна пока работает Colab сессия

ИНСТРУКЦИЯ ПО ИСПОЛЬЗОВАНИЮ

1️⃣ Дождитесь загрузки модели (2-3 минуты)
2️⃣ Для анализа одного интервью:
   - Заполните тему и текст интервью
   - Нажмите 'Выполнить кодирование'
3️⃣ Для пакетного анализа:
   - Загрузите CSV файл с колонками: topic, transcript
   - Нажмите 'Запустить пакетное кодирование'

Внимание: Модель весит ~4GB, загрузка может занять время

GPU Memory allocated: 0.00 GB
GPU Memory reserved: 0.00 GB

Сессия активна. Нажмите STOP в Colab для остановки.
